# 04c — Scaling con RobustScaler

Scala la feature matrix con `RobustScaler` (mediana=0, scala=IQR).
Robusto agli outlier residui che possono sopravvivere al log1p.

Produce:
- `data/feature_matrix_scaled_v1.parquet`
- `data/scaler_v1.joblib`

**Prerequisito**: `data/feature_matrix_graph_v1.parquet` esiste (output di 04b).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib

df = pd.read_parquet("../data/feature_matrix_graph_v1.parquet")
print(f"Caricato: {df.shape}")
print(f"NaN: {df.isnull().sum().sum()} (atteso: 0)")
print(f"Colonne: {list(df.columns)}")

In [ ]:
# Separa agent_id dalle feature
feature_cols = [c for c in df.columns if c != "agent_id"]
print(f"Feature da scalare ({len(feature_cols)}): {feature_cols}")

In [ ]:
# Fitting e trasformazione
scaler = RobustScaler()
scaled_values = scaler.fit_transform(df[feature_cols])

df_scaled = pd.DataFrame(scaled_values, columns=feature_cols)
df_scaled.insert(0, "agent_id", df["agent_id"].values)

print(f"Shape post-scaling: {df_scaled.shape}")
print(f"NaN: {df_scaled.isnull().sum().sum()} (atteso: 0)")

In [ ]:
# Verifica: per RobustScaler, la mediana di ogni feature scalata dovrebbe essere ~0
# e l'IQR ~1 (per definizione)
print("Statistiche post-scaling (campione di feature):")
stats = df_scaled[feature_cols].agg(["median", lambda x: x.quantile(0.75) - x.quantile(0.25)])
stats.index = ["median", "IQR"]
print(stats.round(4).to_string())

In [ ]:
# Statistiche descrittive complete
print("Describe post-scaling:")
print(df_scaled[feature_cols].describe().round(4).to_string())

In [ ]:
# Salvataggio parquet
parquet_path = "../data/feature_matrix_scaled_v1.parquet"
df_scaled.to_parquet(parquet_path, index=False)
print(f"Salvato: {parquet_path}")

# Salvataggio scaler
scaler_path = "../data/scaler_v1.joblib"
joblib.dump(scaler, scaler_path)
print(f"Salvato: {scaler_path}")

In [ ]:
# Verifica idempotenza: ricarica e controlla shape
df_reload = pd.read_parquet(parquet_path)
scaler_reload = joblib.load(scaler_path)

assert df_reload.shape == df_scaled.shape, "Shape mismatch al ricaricamento parquet!"
assert hasattr(scaler_reload, "center_"), "Scaler non correttamente serializzato!"

print(f"Verifica parquet: OK — shape {df_reload.shape}")
print(f"Verifica scaler: OK — center_ shape {scaler_reload.center_.shape}")
print("\nDone.")